# Week 5 · 03 Backpropagation: every gradient, by hand and by code

Training changes the weights to make the loss smaller, and for that it needs the gradient of the loss with respect to **every** weight. Backpropagation computes all of them in one sweep from the loss backwards, using nothing but the chain rule. This notebook starts from the end (one training step with every number shown), then builds it up: the chain rule on small graphs, a 2-2-1 network by hand, a whole layer in matrix form, the algorithm as a card, checking gradients numerically, why deep sigmoid networks stop learning, and finally your own MLP trained on the moons. Reading: ISL §10.7.

**How to use this notebook**
- Run cells one at a time with **Shift + Enter**.
- Before running an example, **guess what it prints**. The real result is written after `# →` in the comments, so you can check yourself.
- **Your turn** cells are for you. Open **Show one answer** only after a real attempt.
- If things get weird, click **Restart** at the top and run from the top again.

## Words you'll see
- **computational graph**: a computation drawn as boxes (operations) joined by arrows (the values passed between them).
- **upstream gradient**: $\partial L / \partial(\text{a box's output})$: it arrives from the part of the graph closer to the loss.
- **local gradient**: $\partial(\text{a box's output}) / \partial(\text{its input})$: known from the box alone, e.g. $\sigma'(z)$.
- **backward pass**: computing every gradient, from the loss back towards the input.
- **learning rate $\eta$**: the step size in the update $\theta \leftarrow \theta - \eta\, \nabla_\theta L$.
- **step**: one update of all the weights. In this notebook every step uses all the data (full batch); `05` uses small batches.

**Run this cell first.** It loads NumPy, the picture helpers (`figures.py` in this folder) and the exercise checker.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from figures import *
import checks; checks.load(globals(), "03")      # defines check(n)

## M. The map: what's in this notebook and why

🖼 **Picture.** From the forward pass of `02` to training. The top row computes gradients for one tiny network; the middle row turns that into an algorithm for any MLP and a way to test it; the bottom row trains one.

In [ ]:
fig_bp_map()

| Section | What | Why it's here |
|---|---|---|
| 0 | recap, and the dense layer's backward pass from memory | §4 is built on it (`01` §8) |
| E | one training step, every number | start from the end: what backprop produces and what the step does with it |
| 1 | why gradients, and why not by brute force | the reason backprop exists |
| 2 | the chain rule on a graph | the only idea backprop uses, on a 3-input example |
| 3 | a 2-2-1 network by hand | the §E numbers, derived one step at a time |
| 4 | a whole layer in matrix form | the same steps for a batch, as matrix products |
| 5 | the algorithm card | backprop for any MLP, in a fixed format |
| 6 | gradient checking | how to know your backward code is right |
| 7 | vanishing gradients | why deep sigmoid networks stop learning, and why ReLU + good initialization help |
| 8–9 | training on the moons; your class | the whole thing from scratch, then as a class with `fit` |

## 0. Recap first · 10 min, closed book

**Quick check:** Chain rule: $\frac{d}{dx} f(g(x)) = $ ?

<details><summary>Answer</summary>

$f'(g(x))\, g'(x)$: the outer derivative at the inner value, times the inner derivative. [Week 1 · 03 §7 ↗](../week1/03_math_essentials.ipynb)

</details>

**Quick check:** For softmax followed by cross-entropy, $\partial L / \partial z = $ ? And for sigmoid with binary cross-entropy?

<details><summary>Answer</summary>

$p - e_y$ in both cases (for the sigmoid: $p - y$, one number). [02 §7 ↗](02_neural_networks.ipynb)

</details>

**Quick check:** $Z = XW + b$ with $X$ $(n, d)$, $W$ $(d, h)$, and $G = \partial L / \partial Z$. What are $\partial L / \partial W$ and $\partial L / \partial b$, and how can the shapes alone tell you?

<details><summary>Answer</summary>

$X^\top G$ $(d, h)$ and the column sums of $G$, $(h,)$. The gradient has the shape of $W$; from $X$ and $G$ the only product with that shape is $X^\top G$. [01 §8 ↗](01_formula_to_numpy.ipynb)

</details>

**Quick check:** The gradient descent update?

<details><summary>Answer</summary>

$\theta \leftarrow \theta - \eta\, \nabla_\theta L$: a small step against the gradient, i.e. downhill. [Week 1 · 03 §7 ↗](../week1/03_math_essentials.ipynb)

</details>

**Quick check:** $\sigma'(z)$ in terms of $\sigma(z)$, and its largest value?

<details><summary>Answer</summary>

$\sigma(z)(1 - \sigma(z))$; at most $0.25$ (at $z = 0$). It's why §7 happens.

</details>

**Exercise 1 · Dense layer backward, from memory.** For $Z = XW + b$ and $G = \partial L / \partial Z$: return `(dX, dW, db)` $= (G W^\top,\ X^\top G,\ \text{column sums of } G)$, shapes `(n, d)`, `(d, h)`, `(h,)`.

**Example**: what goes in, and what should come out:
```python
X = np.array([[1.0, 2.0]]); W = np.array([[1.0], [0.5]]); G = np.array([[2.0]])     # n = 1, d = 2, h = 1
dX, dW, db = dense_backward(X, W, G)
dX    # → [[2. 1.]]
dW    # → [[2.]
      #    [4.]]
db    # → [2.]
```

In [ ]:
def dense_backward(X, W, G):
    pass  # your code

In [ ]:
check(1)

## E. Start from the end: one training step, every number

A tiny network: 2 inputs, 2 hidden units with sigmoid, 1 output with sigmoid, binary cross-entropy loss. One training example, $x = (1, 2)$ with label $y = 1$. Backpropagation's product is the **gradient** column below: one number per parameter, the same shapes as the parameters. A gradient descent step with $\eta = 0.5$ then gives the last column.

| Parameter | Before | $\partial L / \partial(\text{parameter})$ | After: before $- 0.5 \times$ gradient |
|---|---|---|---|
| $W_1$ $(2, 2)$ | $\begin{pmatrix} 0.1 & 0.3 \\ 0.2 & -0.1 \end{pmatrix}$ | $\begin{pmatrix} -0.0429 & 0.0565 \\ -0.0858 & 0.1130 \end{pmatrix}$ | $\begin{pmatrix} 0.1215 & 0.2717 \\ 0.2429 & -0.1565 \end{pmatrix}$ |
| $b_1$ $(2,)$ | $(0,\ 0.1)$ | $(-0.0429,\ 0.0565)$ | $(0.0215,\ 0.0717)$ |
| $w_2$ $(2,)$ | $(0.4,\ -0.5)$ | $(-0.2842,\ -0.2511)$ | $(0.5421,\ -0.3745)$ |
| $b_2$ | $0.2$ | $-0.4566$ | $0.4283$ |

**What the step did:** before it, the network said $p = 0.5434$ for this example (loss $0.6099$); after it, $p = 0.6438$ (loss $0.4404$). The prediction moved towards the label $y = 1$. Training is this step repeated over many examples. Each gradient's **sign** says which way to move that weight; its **size** says how much the loss reacts to it.

PyTorch's autograd produces the same gradient table from `loss.backward()` (`06` explains every line):

In [ ]:
import torch
x = torch.tensor([1.0, 2.0], dtype=torch.float64); y = 1.0
W1 = torch.tensor([[0.1, 0.3], [0.2, -0.1]], dtype=torch.float64, requires_grad=True)   # requires_grad: track it for gradients
b1 = torch.tensor([0.0, 0.1], dtype=torch.float64, requires_grad=True)
w2 = torch.tensor([0.4, -0.5], dtype=torch.float64, requires_grad=True)
b2 = torch.tensor(0.2, dtype=torch.float64, requires_grad=True)
p = torch.sigmoid(torch.sigmoid(x @ W1 + b1) @ w2 + b2)
loss = -(y * torch.log(p) + (1 - y) * torch.log(1 - p))
loss.backward()                                    # fills W1.grad, b1.grad, w2.grad, b2.grad
print(round(loss.item(), 4), W1.grad.numpy().round(4).tolist(), w2.grad.numpy().round(4).tolist(), round(b2.grad.item(), 4))
# → 0.6099 [[-0.0429, 0.0565], [-0.0858, 0.113]] [-0.2842, -0.2511] -0.4566

**The rest of this notebook answers:** how is the gradient column computed? By hand in §3, for a whole batch in §4–5, and by your own code in §8–9.

## 1. Why gradients, and why not by brute force

**The question:** we want weights that make the loss small. Gradient descent repeats $\theta \leftarrow \theta - \eta\, \nabla_\theta L$, which needs $\partial L / \partial \theta_j$ for **every** parameter $\theta_j$ at every step.

**Brute force** would nudge each parameter and measure the change: $\frac{L(\theta + \varepsilon e_j) - L(\theta - \varepsilon e_j)}{2\varepsilon}$. That's 2 forward passes **per parameter**. For `02`'s MNIST network (109,386 parameters) that's 218,772 forward passes for one step, and training takes thousands of steps. **Backpropagation** gets all 109,386 gradients from one forward pass plus one backward pass, which costs about as much as two or three forward passes. That's the whole reason it exists, and it is why deep learning is possible at all.

Brute force is still useful for one thing: **checking** a backward implementation on a tiny network (§6).

## 2. The chain rule on a graph

**The question:** how do we get $\partial f / \partial(\text{each input})$ for a computation made of many small steps? Draw it as a graph. Example: $f = (x + y)\, z$ with $x = 3$, $y = -1$, $z = 2$. Forward (left to right): $q = x + y = 2$, then $f = q z = 4$.

🖼 **Picture.** Forward values in black; gradients in red, computed right to left. Start with $\partial f / \partial f = 1$ and move back one box at a time.

In [ ]:
fig_chain_graph()

**The one rule:** at every box, **downstream gradient = upstream gradient × local gradient**. Each box only needs its own local derivative; it never needs to know the rest of the graph.

| Box | Forward | Local gradients | Upstream | Downstream (= upstream × local) |
|---|---|---|---|---|
| $f = q \cdot z$ | $2 \cdot 2 = 4$ | $\partial f / \partial q = z = 2$, $\ \partial f / \partial z = q = 2$ | 1 | $\partial f / \partial q = 2$, $\ \partial f / \partial z = 2$ |
| $q = x + y$ | $3 - 1 = 2$ | $\partial q / \partial x = 1$, $\ \partial q / \partial y = 1$ | $\partial f / \partial q = 2$ | $\partial f / \partial x = 2$, $\ \partial f / \partial y = 2$ |

Three box types cover most networks, plus one rule for a value used twice:

🖼 **Picture.** **Add** passes the upstream gradient unchanged to both inputs. **Multiply** gives each input the upstream gradient times the **other** input. **Max** sends it all to the larger input (this is ReLU: $\max(0, z)$ passes the gradient if $z > 0$, else 0). **Fan-out**: if a value feeds several boxes, the gradients from all of them **add up**.

In [ ]:
fig_gates()

**Why the gradients add at a fan-out** (the multivariable chain rule, from calculus): if $f$ depends on $x$ only through $u_1, \dots, u_m$, then $\frac{\partial f}{\partial x} = \sum_{j=1}^{m} \frac{\partial f}{\partial u_j}\, \frac{\partial u_j}{\partial x}$. Each path from $x$ to $f$ contributes its own product, and the contributions add. Example: $f = xy + x$ at $x = 3$, $y = 2$: through $u = xy$ the path gives $1 \cdot y = 2$; through $v = x$ it gives $1 \cdot 1 = 1$; so $\partial f / \partial x = 3$. Checked numerically:

In [ ]:
f = lambda x, y: x * y + x
eps = 1e-6
print(round((f(3 + eps, 2) - f(3 - eps, 2)) / (2 * eps), 6), round((f(3, 2 + eps) - f(3, 2 - eps)) / (2 * eps), 6))   # → 3.0 3.0

This is why the bias gradient of a layer is a **sum** over the batch (`01` §8): the same $b$ is used for every sample, a fan-out.

## 3. A 2-2-1 network by hand

**The forward pass**, for one example (no batch: $x$ is a vector), with $W_1$ $(2, 2)$, $b_1$ $(2,)$, $w_2$ $(2,)$, $b_2$ a number:

$$
z_1 = x W_1 + b_1, \quad h = \sigma(z_1), \quad z_2 = h \cdot w_2 + b_2, \quad p = \sigma(z_2), \quad L = -[y \log p + (1 - y)\log(1 - p)]
$$

With §E's numbers: $z_1 = (1 \cdot 0.1 + 2 \cdot 0.2 + 0,\ 1 \cdot 0.3 - 2 \cdot 0.1 + 0.1) = (0.5, 0.2)$, $h = (0.6225, 0.5498)$, $z_2 = 0.4 \cdot 0.6225 - 0.5 \cdot 0.5498 + 0.2 = 0.1741$, $p = 0.5434$, $L = -\log 0.5434 = 0.6099$.

**The backward pass**, from the loss towards the input. Each line uses the line above it (the upstream gradient) and one local derivative:

| Step | Quantity | Formula (upstream × local) | Value |
|---|---|---|---|
| 1 | $\partial L / \partial z_2$ | $p - y$ (sigmoid + BCE, `02` §7) | $0.5434 - 1 = -0.4566$ |
| 2 | $\partial L / \partial w_2$, $\partial L / \partial b_2$ | $\frac{\partial L}{\partial z_2} \cdot h$, $\ \frac{\partial L}{\partial z_2} \cdot 1$ (multiply box, add box) | $(-0.2842, -0.2511)$, $\ -0.4566$ |
| 3 | $\partial L / \partial h$ | $\frac{\partial L}{\partial z_2} \cdot w_2$ (multiply box: the other input) | $-0.4566 \cdot (0.4, -0.5) = (-0.1826, 0.2283)$ |
| 4 | $\partial L / \partial z_1$ | $\frac{\partial L}{\partial h} \odot \sigma'(z_1) = \frac{\partial L}{\partial h} \odot h(1 - h)$ | $(-0.1826 \cdot 0.2350,\ 0.2283 \cdot 0.2475) = (-0.0429, 0.0565)$ |
| 5 | $\partial L / \partial W_1$, $\partial L / \partial b_1$ | $x^\top \frac{\partial L}{\partial z_1}$ (an outer product: $x_j \cdot \frac{\partial L}{\partial z_{1,k}}$), $\ \frac{\partial L}{\partial z_1}$ | $\begin{pmatrix} -0.0429 & 0.0565 \\ -0.0858 & 0.1130 \end{pmatrix}$, $\ (-0.0429, 0.0565)$ |

🖼 **Picture.** The same numbers on the network. Each weight's gradient is (the value flowing **into** that edge) × ($\partial L / \partial z$ of the unit it feeds): e.g. the edge $x_2 \to h_1$ has gradient $2 \times (-0.0429) = -0.0858$.

In [ ]:
fig_bp_221_graph()

**Read the signs.** $\partial L / \partial w_{2,1} = -0.2842 < 0$: increasing that weight lowers the loss, so gradient descent increases it ($0.4 \to 0.5421$). That makes sense: $h_1 > 0$ and we want $p$ larger. $\partial L / \partial w_{2,2}$ is negative too, so $w_{2,2}$ also goes up ($-0.5 \to -0.3745$).

The five steps in NumPy, five lines:

In [ ]:
sigmoid = lambda z: 1 / (1 + np.exp(-z))
x, y = np.array([1.0, 2.0]), 1.0
W1, b1 = np.array([[0.1, 0.3], [0.2, -0.1]]), np.array([0.0, 0.1])
w2, b2 = np.array([0.4, -0.5]), 0.2
h = sigmoid(x @ W1 + b1); p = sigmoid(h @ w2 + b2)           # forward, keeping h

d2 = p - y                       # step 1
dw2, db2 = d2 * h, d2            # step 2
dh = d2 * w2                     # step 3
dz1 = dh * h * (1 - h)           # step 4
dW1, db1 = np.outer(x, dz1), dz1 # step 5: np.outer(a, b)[j, k] = a[j] * b[k]
print(dW1.round(4).tolist(), db1.round(4), dw2.round(4), round(db2, 4))
# → [[-0.0429, 0.0565], [-0.0858, 0.113]] [-0.0429  0.0565] [-0.2842 -0.2511] -0.4566

**Drill (on paper).** Same network and input, but the label is $y = 0$. Compute $\partial L / \partial z_2$, $\partial L / \partial w_2$, $\partial L / \partial z_1$ and $\partial L / \partial W_1$. (Only step 1 changes; everything after it is scaled.)

<details><summary>Worked answer</summary>

Step 1: $p - y = 0.5434$. Step 2: $\partial L / \partial w_2 = 0.5434 \cdot (0.6225, 0.5498) = (0.3382, 0.2988)$. Step 3: $\partial L / \partial h = 0.5434 \cdot (0.4, -0.5) = (0.2174, -0.2717)$. Step 4: $\partial L / \partial z_1 = (0.2174 \cdot 0.2350,\ -0.2717 \cdot 0.2475) = (0.0511, -0.0673)$. Step 5: $\partial L / \partial W_1 = \begin{pmatrix} 0.0511 & -0.0673 \\ 0.1022 & -0.1345 \end{pmatrix}$. Every sign flipped compared with $y = 1$: now the step pushes $p$ down.

</details>

**Exercise 2 · Backprop for the 2-layer network.** Sigmoid hidden layer, sigmoid output, binary cross-entropy, **one** example: `x` `(d,)`, `y` 0.0 or 1.0, `W1` `(d, h)`, `b1` `(h,)`, `W2` `(h,)`, `b2` a float. Return `(dW1, db1, dW2, db2)` with the parameters' shapes, following steps 1–5.

**Example**: what goes in, and what should come out:
```python
dW1, db1, dW2, db2 = backprop_221(np.array([1.0, 2.0]), 1.0, np.array([[0.1, 0.3], [0.2, -0.1]]),
                                  np.array([0.0, 0.1]), np.array([0.4, -0.5]), 0.2)
dW1    # → [[-0.0429  0.0565]
       #    [-0.0858  0.113 ]]
dW2    # → [-0.2842 -0.2511]
db2    # → -0.4566
```

In [ ]:
def backprop_221(x, y, W1, b1, W2, b2):
    pass  # your code

In [ ]:
check(2)

## 4. A whole layer, for a whole batch

**The question:** §3 handled one example and one unit at a time. For a batch $X$ $(n, d)$ and a layer of $h$ units, the same steps become matrix products. A layer has two parts, and each has its own backward rule.

> ↩ **Recap** · [01 §8 (the dense part) ↗](01_formula_to_numpy.ipynb)
>
> $Z = H_{\text{prev}} W + b$. Given $G = \partial L / \partial Z$ $(n, h)$: $\partial L / \partial W = H_{\text{prev}}^\top G$, $\partial L / \partial b = $ column sums of $G$, $\partial L / \partial H_{\text{prev}} = G W^\top$.

**Proposition (the activation part).** If $H = f(Z)$ is applied entry by entry, then $\frac{\partial L}{\partial Z} = \frac{\partial L}{\partial H} \odot f'(Z)$, where $\odot$ is the entry-by-entry product.

**Proof.** $z_{ik}$ affects only $h_{ik} = f(z_{ik})$ and nothing else, so the chain rule has a single term: $\frac{\partial L}{\partial z_{ik}} = \frac{\partial L}{\partial h_{ik}}\, f'(z_{ik})$. $\blacksquare$

**Remarks.** (1) It's `*` in NumPy, not `@`: no sum, because there's no fan-out. (2) For ReLU, $f'(Z)$ is the mask `Z > 0`: the gradient passes where the unit was active and is blocked where it wasn't. (3) For the sigmoid, $f'(Z) = H(1 - H)$ can be computed from the stored $H$, which is why the forward pass keeps it.

| Backward of one layer | Formula | Shape |
|---|---|---|
| arriving: $\partial L / \partial H$ | from the layer above | $(n, h)$ |
| through the activation | $G = \frac{\partial L}{\partial H} \odot f'(Z)$ | $(n, h)$ |
| weights | $H_{\text{prev}}^\top G$ | $(d, h)$ |
| biases | $\sum_i G_{i,:}$ | $(h,)$ |
| passed down | $G W^\top$ | $(n, d)$ |

**Exercise 3 · Sigmoid, backward.** Given $\partial L / \partial H$ (`dH`) and the pre-activations `Z` (same shape), return $\partial L / \partial Z = \partial L / \partial H \odot \sigma(Z)(1 - \sigma(Z))$, **same shape**.

**Example**: what goes in, and what should come out:
```python
sigmoid_backward(np.array([[1.0, -2.0]]), np.array([[0.0, 2.0]]))    # → [[ 0.25 -0.21]]   1 × σ'(0) = 0.25;  -2 × σ'(2) = -2 × 0.105
```

In [ ]:
def sigmoid_backward(dH, Z):
    pass  # your code

In [ ]:
check(3)

**Exercise 4 · ReLU, backward.** Return $\partial L / \partial Z = \partial L / \partial H \odot \mathbb{1}[Z > 0]$, **same shape** (gradient 0 where $Z \le 0$).

**Example**: what goes in, and what should come out:
```python
relu_backward(np.array([[1.0, 2.0, 3.0]]), np.array([[-1.0, 0.0, 4.0]]))    # → [[0. 0. 3.]]
```

In [ ]:
def relu_backward(dH, Z):
    pass  # your code

In [ ]:
check(4)

## 5. The algorithm card: backpropagation for an MLP

**Goal.** $\partial L / \partial W_l$ and $\partial L / \partial b_l$ for every layer $l = 1, \dots, L$ of an MLP with ReLU hidden layers and a softmax output, where $L$ is the mean cross-entropy over a batch $X$ $(n, d)$ with labels $y$.

**Steps.**

- **Forward**, storing what backward will need: $H_0 = X$; for $l = 1, \dots, L$: $Z_l = H_{l-1} W_l + b_l$ and $H_l = \text{ReLU}(Z_l)$ (for $l = L$, no ReLU: $Z_L$ are the logits). Keep every $H_{l-1}$ and $Z_l$.
- **Output gradient**: $P = \operatorname{softmax}(Z_L)$ per row, $G_L = \frac{1}{n}(P - Y)$, with $Y$ the one-hot labels (`02` §7).
- **Backward**, for $l = L, L - 1, \dots, 1$: $\ \partial L / \partial W_l = H_{l-1}^\top G_l$, $\ \partial L / \partial b_l = \sum_i (G_l)_{i,:}$, and if $l > 1$: $\ G_{l-1} = (G_l W_l^\top) \odot \mathbb{1}[Z_{l-1} > 0]$.
- **Update** (gradient descent, step $t$): $W_l^{(t+1)} = W_l^{(t)} - \eta\, \partial L / \partial W_l$, the same for $b_l$. Use $G_l W_l^\top$ **before** changing $W_l$.

**Why it works.** Each line of step 3 is the chain rule for one layer (§4): the dense part, then the activation part. Going from the loss backwards, every $G_l$ is computed once and used twice: for this layer's weight gradient and for the next $G$. That reuse (each partial result is computed once, like dynamic programming) is what makes the cost one backward pass instead of one pass per parameter.

**Caveats.** (1) **Memory**: every $H_{l-1}$ and $Z_l$ is kept until the backward pass, which is why training needs far more memory than prediction. (2) **ReLU at exactly 0** has no derivative; use 0 (the mask `Z > 0`). (3) Use a stable softmax (subtract the row maximum). (4) A wrong backward pass often still trains, just badly, so it fails silently. **Always gradient-check a new implementation** (§6).

**Cost.** The forward pass costs about $n \sum_l d_{l-1} d_l$ multiplications (one matrix product per layer). The backward pass does two matrix products per layer, so about twice that.

**Example trace.** §3's table is this algorithm with $L = 2$, one example, sigmoids instead of ReLU, and a sigmoid output instead of softmax.

## 6. Gradient checking: is my backward pass right?

**The question:** backward code has many places to go wrong (a missing transpose, the wrong mask, a forgotten $1/n$), and a wrong gradient often still trains, just worse. How do we test it? Compare with a numerical derivative on a small network:

$$
\frac{\partial L}{\partial \theta_j} \approx \frac{L(\theta + \varepsilon e_j) - L(\theta - \varepsilon e_j)}{2\varepsilon}
$$

**Why the centered version** (rather than $\frac{L(\theta + \varepsilon e_j) - L(\theta)}{\varepsilon}$)? Its error shrinks like $\varepsilon^2$ instead of $\varepsilon$: by Taylor's theorem the $\varepsilon^1$ error terms of the two sides cancel.

**Choosing $\varepsilon$: too large or too small both fail.** Measured on §3's network, for $\partial L / \partial W_{1, (0,0)} = -0.0429204070$:

| $\varepsilon$ | centered estimate | error | why |
|---|---|---|---|
| 1e-1 | −0.0428938057 | 2.7e-05 | too large: the curve isn't straight over that width |
| 1e-3 | −0.0429204043 | 2.7e-09 |  |
| 1e-5 | −0.0429204070 | 2.1e-12 | about right |
| 1e-7 | −0.0429204072 | 2.6e-10 | rounding starts to show |
| 1e-9 | −0.0429203340 | 7.3e-08 | too small: the two losses agree in almost every digit, |
| 1e-13 | −0.0432986980 | 3.8e-04 | so their difference is mostly rounding error |

🖼 **Picture.** The same experiment for many $\varepsilon$. Right of the sweet spot the error is the Taylor approximation's; left of it, rounding (about $10^{-16} / \varepsilon$, since float64 keeps about 16 digits) takes over.

In [ ]:
fig_eps_tradeoff()

**How to compare.** Use the relative error $\frac{\lvert g_{\text{backprop}} - g_{\text{numeric}} \rvert}{\max(\lvert g_{\text{backprop}} \rvert, \lvert g_{\text{numeric}} \rvert)}$. In float64 with $\varepsilon = 10^{-5}$: below $10^{-7}$ is right; around $10^{-4}$ or larger means a bug. (In float32, as on GPUs, use $\varepsilon \approx 10^{-3}$ and expect only about $10^{-4}$.) Avoid points where a ReLU input is near 0: the kink makes the numerical derivative meaningless there.

> ⚠️ **Common mistake.** Writing `theta[j] += eps`, evaluating, and forgetting to put the old value back. Your function then **changes the caller's parameters**: the same in-place trap as `update_weights` in `00` A2. Save the old value and restore it.

**Exercise 5 · A numerical gradient.** Return an array shaped like `theta` whose entry at each position is $\frac{f(\theta + \varepsilon e) - f(\theta - \varepsilon e)}{2\varepsilon}$ ($e$ = 1 at that position, 0 elsewhere). `f` takes an array like `theta` and returns a float. `theta` may have **any shape** (`for idx in np.ndindex(theta.shape):` visits every position as a tuple). Afterwards `theta` must hold its original values (the check verifies).

**Example**: what goes in, and what should come out:
```python
f = lambda t: float((t ** 2).sum())          # gradient = 2t
theta = np.array([[1.0, -2.0], [0.5, 3.0]])  # (2, 2)
numerical_grad(f, theta)
# → [[ 2. -4.]
#    [ 1.  6.]]    (up to about 1e-10)
theta   # → unchanged
```

In [ ]:
def numerical_grad(f, theta, eps=1e-5):
    pass  # your code

In [ ]:
check(5)

## 7. Vanishing (and exploding) gradients

**The question:** why couldn't people train deep networks with sigmoids for decades, and what changed?

**Proposition.** Take a chain of $L$ layers with one sigmoid unit each: $h_l = \sigma(w_l h_{l-1})$. Then

$$
\left\lvert \frac{\partial h_L}{\partial h_0} \right\rvert = \prod_{l=1}^{L} \lvert w_l \rvert\, \sigma'(w_l h_{l-1}) \;\le\; \prod_{l=1}^{L} \frac{\lvert w_l \rvert}{4}.
$$

**Proof.** By the chain rule, $\frac{\partial h_L}{\partial h_0} = \prod_{l=1}^{L} \frac{\partial h_l}{\partial h_{l-1}}$, and each factor is $\frac{\partial h_l}{\partial h_{l-1}} = \sigma'(w_l h_{l-1})\, w_l$. Since $0 < \sigma'(z) \le \frac{1}{4}$ for every $z$, the bound follows. $\blacksquare$

**Remarks.** (1) With weights of size at most 1, the gradient that reaches the first layer is at most $4^{-L}$: for $L = 10$, $9.5 \times 10^{-7}$. The first layers barely learn. (2) If the weights are large instead, the same product can **explode**; the usual fix is gradient clipping (cap the gradient's size). (3) Wide layers behave the same way with matrices in place of the $w_l$. (4) What fixed it: **ReLU**, whose derivative is exactly 1 wherever the unit is active; **initialization** scaled to the layer size (He or Xavier), which keeps the signal's size steady; and later **batch normalization** and **residual connections** (Weeks 6–7).

🖼 **Picture.** Measured on a 10-layer network with 32 units per layer, at initialization: the size of $\partial L / \partial W$ for each layer. With sigmoids it falls about 4–5× per layer going back towards the input; with ReLU and He initialization it stays roughly level.

In [ ]:
fig_vanishing()

## 8. Training an MLP on the moons, from scratch

Everything together: §5's algorithm card in about 20 lines. Data: 200 points of the two moons (from `figures.py`), labels 0/1. Network $2 \to 16 \to 16 \to 2$ with ReLU, He initialization, full-batch gradient descent with $\eta = 0.5$ for 2000 steps.

In [ ]:
def softmax_rows(Z):                                     # 01 exercise 1
    E = np.exp(Z - Z.max(axis=1, keepdims=True))
    return E / E.sum(axis=1, keepdims=True)

X_m, y_m = moons(200, 0.2, 0)                            # (200, 2) floats, (200,) ints 0/1
sizes = [2, 16, 16, 2]
rng = np.random.default_rng(0)
Ws = [rng.normal(size=(a, b)) * np.sqrt(2 / a) for a, b in zip(sizes[:-1], sizes[1:])]   # He initialization
bs = [np.zeros(b) for b in sizes[1:]]
lr, losses = 0.5, []
for step in range(2000):
    Hs, Zs = [X_m], []                                   # step 1: forward, keeping every layer's input and sum
    for l in range(len(Ws)):
        Z = Hs[-1] @ Ws[l] + bs[l]
        Zs.append(Z)
        Hs.append(np.maximum(Z, 0) if l < len(Ws) - 1 else Z)
    P = softmax_rows(Hs[-1])
    losses.append(-np.log(P[np.arange(len(y_m)), y_m]).mean())
    G = (P - np.eye(2)[y_m]) / len(y_m)                  # step 2: the output gradient
    for l in range(len(Ws) - 1, -1, -1):                 # step 3: backward, last layer first
        dW, db = Hs[l].T @ G, G.sum(axis=0)
        if l > 0:
            G = (G @ Ws[l].T) * (Zs[l - 1] > 0)          # uses Ws[l] BEFORE it is updated
        Ws[l] -= lr * dW                                 # step 4: update (in place, on purpose: these are our own arrays)
        bs[l] -= lr * db
print(round(losses[0], 3), round(losses[100], 3), round(losses[-1], 3), (P.argmax(axis=1) == y_m).mean())   # → 0.893 0.101 0.024 0.98

🖼 **Picture.** Left: the loss over the 2000 steps (log scale): fast at first, then slower. Right: the trained network's regions. This is **training** accuracy; whether it generalizes to new points is Week 6's question (validation, regularization).

In [ ]:
fig_moons_training()

**Recipe (training from scratch).** Initialize randomly (He); repeat: forward keeping $H$ and $Z$, loss, $G = (P - Y)/n$, backward layer by layer, update. Watch the loss go down; if it goes up or becomes `nan`, the learning rate is too large (`05` §1).

## 9. Your MLP class, with `fit`

First the backward pass as a function (the loop in §8, steps 1–3), then the class from `02` with a `fit` method that repeats gradient steps.

**Exercise 6 · The gradients of an MLP.** ReLU hidden layers, softmax output, **mean** cross-entropy. `Ws`, `bs` are lists like `02`'s; `X` `(n, d)`; `y` `(n,)` integer labels. Return `(dWs, dbs)`: two lists with the same shapes as `Ws` and `bs`. Don't change `Ws` or `bs`.

**Example**: what goes in, and what should come out:
```python
Ws = [np.array([[1.0, -1.0], [0.5, 2.0]]), np.array([[1.0, 0.0], [-1.0, 1.0]])]    # 2 -> 2 -> 2
bs = [np.zeros(2), np.zeros(2)]
dWs, dbs = mlp_gradients(np.array([[1.0, 1.0]]), np.array([0]), Ws, bs)
dWs[1]    # → [[-0.934  0.934]
          #    [-0.622  0.622]]
dbs[1]    # → [-0.622  0.622]
```

In [ ]:
def mlp_gradients(X, y, Ws, bs):
    pass  # your code

In [ ]:
check(6)

**Check it against PyTorch's autograd** (after exercise 6 passes) on a random network, with the same weights transposed into `nn.Linear`:

In [ ]:
import torch.nn as nn
rng2 = np.random.default_rng(3)
Xc, yc = rng2.normal(size=(5, 3)), np.array([0, 2, 1, 1, 0])
Wc = [rng2.normal(size=(3, 4)), rng2.normal(size=(4, 3))]; bc = [rng2.normal(size=4), rng2.normal(size=3)]
dWs, dbs = mlp_gradients(Xc, yc, Wc, bc)

net = nn.Sequential(nn.Linear(3, 4), nn.ReLU(), nn.Linear(4, 3)).double()     # .double(): float64, like NumPy
with torch.no_grad():
    for layer, W, b in zip([net[0], net[2]], Wc, bc):
        layer.weight.copy_(torch.tensor(W.T)); layer.bias.copy_(torch.tensor(b))
nn.functional.cross_entropy(net(torch.tensor(Xc)), torch.tensor(yc)).backward()   # mean cross-entropy, then backprop
print(np.allclose(dWs[0], net[0].weight.grad.numpy().T), np.allclose(dbs[1], net[2].bias.grad.numpy()))   # → True True

**Exercise 7 · Your MLP class with fit.** `MLPScratch(sizes, seed=0)`: `self.W`, `self.b` as in `02`'s `MyMLP` (He initialization, zero biases), plus `forward`, `predict_proba` and `predict`. `fit(X, y, epochs, lr)`: for each epoch, append the **current** mean cross-entropy to `self.loss_history_` (start it as an empty list inside `fit`), then take one gradient step on all the data with `mlp_gradients`. Return `self`. Don't re-initialize the weights inside `fit`.

**Example**: what goes in, and what should come out:
```python
model = MLPScratch([2, 16, 16, 2], seed=0).fit(X_m, y_m, epochs=2000, lr=0.5)
model.loss_history_[0], model.loss_history_[-1]    # → about 0.89 and 0.02, as in §8
(model.predict(X_m) == y_m).mean()                  # → about 0.98 to 0.99
```

In [ ]:
class MLPScratch:
    def __init__(self, sizes, seed=0):
        pass  # your code

    def forward(self, X):
        pass  # your code

    def predict_proba(self, X):
        pass  # your code

    def predict(self, X):
        pass  # your code

    def fit(self, X, y, epochs=100, lr=0.1):
        pass  # your code

In [ ]:
check(7)

After exercise 7 passes, train your class on the moons:

In [ ]:
model = MLPScratch([2, 16, 16, 2], seed=0).fit(X_m, y_m, epochs=2000, lr=0.5)
print(model.loss_history_[-1] < 0.1, (model.predict(X_m) == y_m).mean() >= 0.95)   # → True True

**Recipe (an MLP class).** `__init__` builds the weights; `forward` is the dense layers; `fit` loops: record the loss, get the gradients, step. The same structure is in every PyTorch training script you'll read (`06` §5).

In [ ]:
for n in range(1, 8):
    check(n)

---
Solutions: `solutions/03_solutions.py`. Next: `04_micrograd.ipynb`, an engine that does backprop for any formula automatically.